# 14. Project - Auto-Acknowledgement Emails for New Tickets

**Goal:** every few hours, find new unassigned tasks and email each reporter: *"We received your ticket, here is the number, our team is on it."*

![Auto-acknowledgement flow](images/14-ticket-acknowledgement-flow.png)

The finished script is also in [`projects/ticket_acknowledgement.py`](projects/ticket_acknowledgement.py), ready for cron.

### What You Need

| Setting | Environment variable |
|---|---|
| Jira site, email and API token | `JIRA_SERVER`, `JIRA_EMAIL`, `JIRA_API_TOKEN` |
| Gmail address that sends the mail | `SENDER_EMAIL` |
| Gmail **app password** (Google Account > Security > 2-Step Verification > App passwords) | `GMAIL_APP_PASSWORD` |
| Optional copy of every email | `SUPPORT_CC` |

## Step 1: Find the New Tickets

In [1]:
import os
from jira import JIRA

jira = JIRA(
    server=os.environ["JIRA_SERVER"],                 # https://your-domain.atlassian.net
    basic_auth=(os.environ["JIRA_EMAIL"], os.environ["JIRA_API_TOKEN"]),
)

In [2]:
jql = "project = MYP AND issuetype = Task AND created >= -3h AND assignee is EMPTY AND resolution = Unresolved"
issue_in_project = jira.search_issues(jql)
total = len(issue_in_project)
print("New tickets:", total)

The JQL picks **unresolved, unassigned tasks created in the last 3 hours**, so a cron job every 3 hours sees each ticket once.

## Step 2: Collect the Details

In [3]:
tnumber = []      # ticket numbers
uname = []        # reporter names
eaddress = []     # reporter emails
subdetails = []   # email subjects

for issue in issue_in_project:
    tnumber.append(issue.key)
    uname.append(issue.fields.reporter.displayName)
    # the email is hidden for some accounts, so read it safely
    eaddress.append(getattr(issue.fields.reporter, "emailAddress", None))
    subdetails.append(issue.key + ": " + issue.fields.summary)

print(tnumber, uname)

> `getattr(obj, "emailAddress", None)` returns `None` instead of crashing when Jira hides a user's email. The original version of this script used `issues.fields.reporter.emailAddress` directly and failed on such accounts.

## Step 3: Send the Emails

In [4]:
import os
import smtplib
import time
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

my_mail = os.environ["SENDER_EMAIL"]
password = os.environ["GMAIL_APP_PASSWORD"]
cc = os.environ.get("SUPPORT_CC")

if total == 0:
    print("No new Jira tickets in the last 3 hours")
else:
    print("JIRA found")
    with smtplib.SMTP("smtp.gmail.com", 587) as connection:
        connection.starttls()
        connection.login(user=my_mail, password=password)

        for i in range(total):
            if not eaddress[i]:
                print("No email address for", uname[i], "- skipped")
                continue

            msg = MIMEMultipart()
            msg["Subject"] = subdetails[i]
            msg["From"] = my_mail
            msg["To"] = eaddress[i]
            if cc:
                msg["Cc"] = cc

            body = (
                f"<p>Dear {uname[i]},<br><br>"
                "Thank you for writing to us.<br><br>"
                f"This email confirms that we have received your Jira ticket <b>{tnumber[i]}</b>. "
                "Please send the approval from your reporting manager. Our team is looking into it "
                "and will get back to you soon.<br><br>"
                "Thank you,<br>CloudOps Support Team</p>"
            )
            msg.attach(MIMEText(body, "html"))
            connection.send_message(msg)
            print("mail has been sent", uname[i])
            time.sleep(3)          # be gentle with the mail server

JIRA found
mail has been sent Ahmad Hassan
mail has been sent Hassan


Improvements over the first version of this script:

- One SMTP connection for all emails instead of reconnecting for each one.
- Port **587** with `starttls()` stated explicitly.
- Credentials from environment variables, never in the code.
- Tickets without a visible email are skipped instead of crashing the run.

## Step 4: Schedule It

```bash
crontab -e
# every 3 hours, on the hour
0 */3 * * * cd /opt/jira-automation && /usr/bin/python3 ticket_acknowledgement.py >> ack.log 2>&1
```

Cron runs with a minimal environment, so put the `export` lines in a file (for example `/opt/jira-automation/.env.sh`, readable only by you) and start the command with `. /opt/jira-automation/.env.sh && ...`.

> **Avoid duplicates:** if the job runs more often than the JQL window (say every hour with `-3h`), reporters get several emails. Keep the window and the schedule equal, or add a label such as `acknowledged` to each issue after emailing and exclude it in the JQL with `labels != acknowledged`.